# Difference-in-Differences: The Card-Krueger Minimum Wage Study

**Question:** When a state raises its minimum wage, does it actually raise workers' pay, and
does it cost jobs?

**Data:** The classic Card & Krueger (1994) natural experiment. In April 1992, New Jersey raised
its minimum wage from the federal floor of \$4.25 to \$5.05/hour. Pennsylvania did not change its
minimum wage over the same period. Card and Krueger surveyed fast-food restaurants in both states
before (February 1992) and after (November 1992) the law change, recording wages, staffing
levels, and store characteristics. This is panel data: each store appears twice.

**Design:** Difference-in-differences (DiD). We compare the *change* in outcomes in New Jersey
(treated) to the *change* in outcomes in Pennsylvania (control) over the same period. As long as
the two states would have followed similar trends absent the policy (the "parallel trends"
assumption), any *extra* change in New Jersey can be attributed to the minimum wage law.

$$Outcome_i = \beta_0 + \beta_1 NJ_i + \beta_2 After_i + \beta_3 (NJ_i \times After_i) + u_i$$

$\beta_3$, the coefficient on the interaction term, is the causal estimate of interest.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option('display.float_format', lambda x: f'{x:.3f}')

df = pd.read_stata('../data/minimum_wage.dta')
df['nj'] = df['state']
df['after'] = df['time']
df['did'] = df['nj'] * df['after']
df['emp_total'] = df['empft'] + df['emppt'] + df['nmgrs']

print(f"{df.shape[0]} store-period observations ({df['store'].nunique()} unique stores)")

820 store-period observations (410 unique stores)


## Table 1: Balance table (baseline, February 1992 only)

Before trusting the comparison, check whether NJ and PA restaurants looked similar *before* the policy change.

In [2]:
baseline = df[df['time'] == 0]
covars = ['wage_st', 'inctime', 'firstinc', 'freemeals', 'empft', 'emppt', 'nmgrs',
          'hoursopen', 'priceentree', 'nregisters']

rows = []
for var in covars:
    pa = baseline[baseline['nj'] == 0][var].dropna()
    nj = baseline[baseline['nj'] == 1][var].dropna()
    tstat, pval = stats.ttest_ind(nj, pa, equal_var=False)
    rows.append({'Variable': var, 'PA Mean': pa.mean(), 'NJ Mean': nj.mean(),
                 'Difference': nj.mean() - pa.mean(), 'P-value': pval})

balance_table = pd.DataFrame(rows).set_index('Variable')
balance_table

,PA Mean,NJ Mean,Difference,P-value
Variable,,,,
wage_st,4.630,4.612,-0.018,0.689
inctime,18.872,18.174,-0.698,0.672
firstinc,0.208,0.229,0.022,0.100
freemeals,0.063,0.215,0.151,0.000
empft,10.076,7.607,-2.469,0.058
emppt,19.487,18.675,-0.812,0.508
nmgrs,3.494,3.341,-0.153,0.290
hoursopen,14.525,14.418,-0.107,0.770
priceentree,1.215,1.348,0.133,0.098


Most baseline characteristics are close between the two states and not statistically distinguishable, average starting wages are within a few cents (\$4.61 NJ vs \$4.63 PA). A few differences do stand out (free meals, full-time staffing, register counts), but the overall picture supports treating PA as a reasonable comparison group for NJ.

## Table 2: Change in wages, before vs. after

In [3]:
wage_summary = df.groupby(['nj', 'after'])['wage_st'].agg(['mean', 'min']).reset_index()
wage_summary['State'] = wage_summary['nj'].map({0: 'Pennsylvania', 1: 'New Jersey'})
wage_summary['Period'] = wage_summary['after'].map({0: 'Before (Feb 1992)', 1: 'After (Nov 1992)'})
wage_summary = wage_summary[['State', 'Period', 'mean', 'min']].rename(
    columns={'mean': 'Mean Starting Wage', 'min': 'Minimum Starting Wage'})
wage_summary

,State,Period,Mean Starting Wage,Minimum Starting Wage
0,Pennsylvania,Before (Feb 1992),4.630,4.250
1,Pennsylvania,After (Nov 1992),4.617,4.250
2,New Jersey,Before (Feb 1992),4.612,4.250
3,New Jersey,After (Nov 1992),5.081,5.050


New Jersey's average starting wage jumps from about \$4.61 to \$5.08, and the *minimum* observed wage rises exactly to the new legal floor of \$5.05. Pennsylvania's wages barely move at all over the same period. This is the raw pattern that the DiD regression will formalize.

## Table 3: DiD regression, compensation outcomes

Each outcome gets its own column: starting wage, months to first raise, first raise amount, and whether employees get free meals.

In [4]:
comp_outcomes = {
    'Starting Wage': 'wage_st',
    'Months to First Raise': 'inctime',
    'First Raise Amount': 'firstinc',
    'Free Meals': 'freemeals',
}

comp_models = {name: smf.ols(f'{col} ~ nj + after + did', data=df).fit(cov_type='HC1')
               for name, col in comp_outcomes.items()}

comp_table = pd.DataFrame({
    name: {
        'DiD Effect (β3)': m.params['did'],
        'Std. Error': m.bse['did'],
        'P-value': m.pvalues['did'],
        'N': int(m.nobs),
    }
    for name, m in comp_models.items()
}).T
comp_table

,DiD Effect (β3),Std. Error,P-value,N
Starting Wage,0.482,0.062,0.000,779.000
Months to First Raise,2.021,2.314,0.382,723.000
First Raise Amount,0.017,0.018,0.328,697.000
Free Meals,0.018,0.057,0.750,820.000


The DiD estimate for starting wage is about **\$0.48/hour** and highly significant, the policy raised wages in NJ relative to PA by roughly that amount. None of the other compensation measures (raise timing, raise size, free meals) move significantly, there's no sign that restaurants clawed back the wage increase through other channels.

## Table 4: DiD regression, employment and store outcomes

If higher minimum wages price out low-wage labor, we'd expect to see it here.

In [5]:
store_outcomes = {
    'Total Employment': 'emp_total',
    'Hours Open': 'hoursopen',
    'Price of Entree': 'priceentree',
    'Number of Registers': 'nregisters',
}

store_models = {name: smf.ols(f'{col} ~ nj + after + did', data=df).fit(cov_type='HC1')
                for name, col in store_outcomes.items()}

store_table = pd.DataFrame({
    name: {
        'DiD Effect (β3)': m.params['did'],
        'Std. Error': m.bse['did'],
        'P-value': m.pvalues['did'],
        'N': int(m.nobs),
    }
    for name, m in store_models.items()
}).T
store_table

,DiD Effect (β3),Std. Error,P-value,N
Total Employment,2.511,2.252,0.265,806.000
Hours Open,-0.127,0.512,0.804,809.000
Price of Entree,0.076,0.110,0.488,784.000
Number of Registers,-0.131,0.209,0.530,792.000


None of the four store-outcome estimates are statistically significant. Total employment doesn't
fall, hours don't shrink, entree prices don't jump, and register counts don't change in any
detectable way. This is the famous, and at the time controversial, Card-Krueger result: the
textbook prediction that minimum wage hikes cost jobs did not show up in this data.

## Takeaway

- **Wages:** the policy worked as intended, NJ starting wages rose by about \$0.48/hour relative
  to PA, and the wage floor moved exactly to the new legal minimum.
- **No visible offsetting cuts:** raise timing, raise size, and free-meal benefits didn't move,
  so there's no evidence employers clawed the increase back through other forms of compensation.
- **No visible employment loss:** total staffing, hours, and prices all stayed statistically flat
  relative to Pennsylvania.
- **Caveat:** this all rests on the parallel-trends assumption, that NJ and PA would have moved
  together absent the law. The baseline balance table supports that being a reasonable
  assumption here, but it's an assumption, not something directly testable with only one
  pre-period.